# Initialize feedstock profiles 

This script walks through functions to initialize feedstock profile in RTM ERW amendments. 

Calculation inputs include: 
1. feedstock dict (keys = mineral names; values = weight fractions, sum to 1 or will be re-scaled to 1)
2. pointer to mineral database (for molar volumes and masses)
3. feedstock density 
4. feedstock grain size (single value for now — build in grain size distribution support later)
5. feedstock application rate (t/ha)
6. roughness factor rule
7. mixing depth
8. mixing type
9. depth grid parameters (max depth and number of cells)

And calculates:
1. feedstock specific surface area (constant)
2. feedstock volume fraction (depth-resolved)
3. feedstock bulk surface area (depth-resolved)

A main function that captures the full workflow is at the bottom of the notebook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# 0. Define application rate, diameter, roughness factor
application_rate = 10  # t/ha
diameter = 50  # um
roughness = 20  # Assuming constant roughness

# 1. Fetch the qualitative colormap object
cmap = mpl.colormaps['Dark2']

# 2. Extract and convert all underlying colors to hex
hex_colors = [mpl.colors.to_hex(color) for color in cmap.colors]
CAT4 = hex_colors[:4]
CAT6 = hex_colors[:6]

## 1. Feedstock composition

Two data files are needed to define a feedstock: 
1. feedstock weight fraction dict (key: mineral name; value: weight fraction)
    - weight fractions will be scaled so they sum to one if they don't already
2. mineral database (for molar weights and volumes by mineral).

We use an example from the CrunchFlow database I was using, where molar volume is in $\text{cm}^3 / \text{mol}$ and molar mass is in $\text{g} / \text{mol}$.

This information is necessary to convert between mass and volume, as well as specific and bulk surface areas. 

Worked example: a 6-mineral basalt composition (real values, pulled from `~/crunch/model_tools/feedstocks/basalt.dat` and `datacom_EW.dbs`). To try a single-mineral feedstock instead, just swap `FEEDSTOCK_COMPOSITION` and `MINERAL_PROPERTIES` for a one-entry dict — everything downstream is written to handle any number of minerals, including one.

In [ ]:
# --- worked example: basalt feedstock (6 minerals) ---
FEEDSTOCK_COMPOSITION = {
    "Clinochlore-14A": 0.366,
    "An50Ab50AS": 0.196,
    "Epidote": 0.256,
    "Ferroactinolite": 0.116,
    "Quartz": 0.052,
    "Titanite": 0.017,
}
FEEDSTOCK_COMPOSITION = {'forsterite': 1.0}
FEEDSTOCK_DENSITY = 3.2

# molar volume [cm3/mol] and molar mass [g/mol], from the CrunchFlow datacom database
MINERAL_PROPERTIES = {
    "Clinochlore-14A": {"molVolume_cm3_mol": 207.11, "molMass_g_mol": 555.7973},
    "An50Ab50AS":      {"molVolume_cm3_mol": 103.14, "molMass_g_mol": 270.2143},
    "Epidote":         {"molVolume_cm3_mol": 139.20, "molMass_g_mol": 483.2227},
    "Ferroactinolite": {"molVolume_cm3_mol": 284.20, "molMass_g_mol": 970.0800},
    "Quartz":          {"molVolume_cm3_mol": 22.688, "molMass_g_mol": 60.0843},
    "Titanite":        {"molVolume_cm3_mol": 48.18,  "molMass_g_mol": 196.0405},
}
MINERAL_PROPERTIES = {'forsterite': {"molVolume_cm3_mol": 43.967, "molMass_g_mol": 140.6931},}

def composition_to_df(composition, mineral_properties):
    """Combine a {mineral: weight_fraction} dict with molar properties into one DataFrame.

    Weight fractions are renormalized to sum to 1 (so callers can pass wt% or
    unnormalized values). Raises if a mineral is missing molar properties.
    """
    missing = set(composition) - set(mineral_properties)
    if missing:
        raise KeyError(f"no molar volume/mass for: {sorted(missing)}")

    total = sum(composition.values())
    df = pd.DataFrame(
        {
            "MINERAL": list(composition.keys()),
            "wt_frac": [v / total for v in composition.values()],
        }
    )
    props = pd.DataFrame(mineral_properties).T.rename_axis("MINERAL").reset_index()
    return df.merge(props, on="MINERAL", validate="one_to_one")


feedstock_df = composition_to_df(FEEDSTOCK_COMPOSITION, MINERAL_PROPERTIES)
feedstock_df

## 2. Surface roughness factor

Real mineral grains aren't smooth spheres — fracture surfaces, etch pits, and
micro-topography give them more reactive surface area than a geometric sphere of
the same diameter would suggest. The **roughness factor** scales geometric
surface area up to account for this, and it is itself a strong function of grain
size (larger grains tend to be relatively rougher per unit geometric area, in the
empirical fits below).

Four options, as a function of grain **radius** `r` [m]:

| Rule | Formula | Notes |
|---|---|---|
| `smooth` | `1.0` | no roughness correction (lower bound) |
| `constant` | `rf` | fixed constant roughness factor      |
| `BM00` | `10^0.7 · r^(-0.1)` | Brantley & Mellott (2000) |
| `B20` | `10^3.3 · r^0.33` | equivalent to Kanzaki et al. (2022) eq. 39 |
|`NSB07` | `20.0` | fixed roughness factor of 20 using the correct interpretation of NSB07 |


In [ ]:
def feedstock_roughness(roughness_option, radius_m, const_rf = None):
    """Roughness factor (dimensionless) for a grain of the given radius [m]."""

    def bm00(r):
        return (10**0.7) * (r**-0.1)

    def b20(r):
        return (10**3.3) * (r**0.33)
    
    def nsb07(r):
        return 20.0

    def smooth(r):
        return 1.0

    rule_map = {"BM00": bm00, "NBS07": nsb07, "B20": b20, "smooth": smooth}

    if roughness_option == "constant":
        if const_rf is None:
            raise ValueError("const_rf must be provided when roughness_option='constant'")
        return const_rf
    if roughness_option in rule_map:
        return rule_map[roughness_option](radius_m)
    try:
        return float(roughness_option)
    except (TypeError, ValueError):
        raise ValueError(f"roughness_option is invalid: {roughness_option!r}")


# sanity check
dustrad = 100 * 1e-6 # [m] 100 microns converted to m
assert np.isclose(feedstock_roughness("smooth", dustrad), 1.0)
print(feedstock_roughness("BM00", dustrad), feedstock_roughness("B20", dustrad), feedstock_roughness("constant", dustrad, const_rf=10))

In [ ]:
# compare roughness rules across the grain-size range typically used for feedstock (1-1000 um diameter)
diam_um_range = np.logspace(0, 3, 100)
radius_m_range = (diam_um_range * 1e-6) / 2
rules = ["smooth", "BM00", "NBS07", "B20"]
rule_colors = dict(zip(rules, CAT4))

fig, ax = plt.subplots(figsize=(7, 4.5))
for rule in rules:
    vals = [feedstock_roughness(rule, r) for r in radius_m_range]
    ax.plot(diam_um_range, vals, label=rule, color=rule_colors[rule], linewidth=2)

# ax.set_xscale("log")
# ax.set_yscale("log")
ax.set_xlabel("grain diameter [µm]")
ax.set_ylabel("roughness factor [-]")
ax.set_title("Roughness factor vs. grain size, by rule")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()

## 3. Convert application rate to mineral surface area 

This calculation requires three steps:
### 1. How much feedstock per unit soil volume
Based in the mixing depth of the soil, amount of soil, and amount of rock, we can solve for the volume fraction of feedstock.

### 2. How much of the feedstock volume goes to each mineral
Convert the weight fraction to a volume fraction

```
vol_frac_in_feedstock = molVolume × (wt_frac / molMass) × dens_feedstock
```

Multiplying by `appRate_volume_per_volume` gives each mineral's volume fraction **per unit soil volume**, which CrunchFlow takes as an initial condition.

### 3. How much surface area for a given mineral volume 
Specific surface area for a sphere is 

$$\text{SSA} = \frac{6}{\rho d}$$

where $\rho$ is density ($m^2\ kg^{-1}$) and $d$ is grain diameter. Then the geometric surface area is scaled by the roughness factor to get the actual input surface area. 

Finally, we can convert the specific surface area to bulk surface area by multiplying by the mineral's mass per unit soil volume (BSA: $m^2$ of mineral surface area per $m^3$ soil).

In [ ]:
def feedstock_mineral_properties(
    feedstock_df, dens_feedstock_g_cm3, appRate_ton_ha, tilldepth_m, diam_um,
    roughness_option="smooth", const_rf=None,
):
    """Per-mineral SSA, BSA, mass, and volume fractions for a well-mixed tilled layer.

    feedstock_df: output of composition_to_df (MINERAL, wt_frac, molVolume_cm3_mol, molMass_g_mol)
    dens_feedstock_g_cm3: bulk density of the feedstock grains [g/cm3]
    appRate_ton_ha: application rate [metric ton/ha]
    tilldepth_m: depth of the well-mixed tilled layer [m]
    diam_um: (single) grain diameter [um]
    roughness_option: passed to feedstock_roughness
    const_rf: fixed roughness factor, required when roughness_option="constant"; passed to feedstock_roughness

    Returns (df, diagnostics) where df is feedstock_df with new columns added,
    and diagnostics holds the scalar intermediates (roughness factor, etc).
    """

    def convert_units(diam_um, dens_g_cm3):
        diam_m = diam_um * 1e-6
        radius_m = diam_m / 2
        dens_kg_m3 = dens_g_cm3 * 1e3
        return diam_m, radius_m, dens_kg_m3

    def volumetric_application_fraction(appRate_ton_ha, dens_kg_m3, tilldepth_m):
        appRate_kg_m2 = appRate_ton_ha * 1e3 / 1e4  # t/ha -> kg/m2
        appRate_vol_per_area = appRate_kg_m2 / dens_kg_m3  # m3 feedstock / m2 soil
        return appRate_vol_per_area / tilldepth_m  # m3 feedstock / m3 soil

    def mineral_ssa_bsa(row, diam_m, roughness_factor, appRate_volume_per_volume):
        mineral_dens_g_cm3 = row["molMass_g_mol"] / row["molVolume_cm3_mol"]
        mineral_dens_kg_m3 = mineral_dens_g_cm3 * 1e3
        ssa_geom = 6 / (mineral_dens_kg_m3 * diam_m)  # m2/kg, smooth sphere
        ssa_total = ssa_geom * roughness_factor  # m2/kg, roughness-corrected

        vol_frac_in_feedstock = (
            row["molVolume_cm3_mol"] * (row["wt_frac"] / row["molMass_g_mol"]) * dens_feedstock_g_cm3
        )
        vol_frac_in_soil = vol_frac_in_feedstock * appRate_volume_per_volume
        mineral_mass = mineral_dens_kg_m3 * vol_frac_in_soil  # kg mineral / m3 soil
        bulk_sa_if_pure = mineral_dens_kg_m3 * ssa_total * appRate_volume_per_volume  # m2/m3 soil
        mineral_bsa = vol_frac_in_feedstock * bulk_sa_if_pure  # m2/m3 soil, actual

        return pd.Series(
            {
                "mineral_SSA_m2_kg": ssa_total,
                "vol_frac_in_feedstock": vol_frac_in_feedstock,
                "vol_frac_in_soil": vol_frac_in_soil,
                "mineral_mass_kg_m3soil": mineral_mass,
                "mineral_BSA_m2_m3soil": mineral_bsa,
            }
        )

    diam_m, radius_m, dens_kg_m3 = convert_units(diam_um, dens_feedstock_g_cm3)
    roughness_factor = feedstock_roughness(roughness_option, radius_m, const_rf=const_rf)
    appRate_volume_per_volume = volumetric_application_fraction(appRate_ton_ha, dens_kg_m3, tilldepth_m)

    out = feedstock_df.copy()
    new_cols = out.apply(
        mineral_ssa_bsa, axis=1, diam_m=diam_m, roughness_factor=roughness_factor,
        appRate_volume_per_volume=appRate_volume_per_volume,
    )
    out = pd.concat([out, new_cols], axis=1)

    diagnostics = {
        "roughness_factor": roughness_factor,
        "appRate_volume_per_volume": appRate_volume_per_volume,
        "diam_m": diam_m,
    }
    return out, diagnostics


# --- worked example ---
mineral_props, diag = feedstock_mineral_properties(
    feedstock_df,
    dens_feedstock_g_cm3 = FEEDSTOCK_DENSITY,  # forsterite density
    appRate_ton_ha       = application_rate,
    tilldepth_m          = 0.3,
    diam_um              = diameter,
    roughness_option     = "constant",
    const_rf             = roughness,
)
print(diag)
mineral_props

In [ ]:
# weight fraction vs. volume fraction: denser accessory minerals (e.g. Titanite) punch
# below their weight; less dense ones (e.g. Quartz) punch above it
order = mineral_props.sort_values("wt_frac", ascending=False)["MINERAL"]
plot_df = mineral_props.set_index("MINERAL").loc[order]

fig, ax = plt.subplots(figsize=(7, 4.5))
y = np.arange(len(plot_df))
h = 0.35
ax.barh(y + h / 2, plot_df["wt_frac"], height=h, label="weight fraction", color=CAT4[0])
ax.barh(y - h / 2, plot_df["vol_frac_in_feedstock"], height=h, label="volume fraction", color=CAT4[1])
ax.set_yticks(y)
ax.set_yticklabels(plot_df.index)
ax.invert_yaxis()
ax.set_xlabel("fraction of feedstock [-]")
ax.set_title("Basalt feedstock: weight fraction vs. volume fraction")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
# which minerals actually dominate reactive surface area (BSA), vs. which dominate mass
bsa_order = mineral_props.sort_values("mineral_BSA_m2_m3soil", ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(bsa_order["MINERAL"], bsa_order["mineral_BSA_m2_m3soil"], color=CAT4[0])
ax.set_ylabel("bulk surface area [m² mineral / m³ soil]")
ax.set_title(f"Mineral BSA at {diag['diam_m']*1e6:.0f} micron grain size, constant roughness (rf={diag['roughness_factor']:.0f})")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

### Sensitivity: grain size, roughness rule, and application rate

Grinding feedstock finer is the single biggest lever on reactive surface area
(BSA scales as roughly `1/diam` before roughness, and the roughness rules add
their own diameter dependence on top). Application rate scales BSA linearly.
The plots below sweep each one-at-a-time, holding the basalt composition and
till depth fixed, to show the relative size of these effects.

In [ ]:
def total_bsa(diam_um, appRate_ton_ha, roughness_option, const_rf=None):
    df, _ = feedstock_mineral_properties(
        feedstock_df, dens_feedstock_g_cm3=2.9, appRate_ton_ha=appRate_ton_ha,
        tilldepth_m=0.2, diam_um=diam_um, roughness_option=roughness_option, const_rf=const_rf,
    )
    return df["mineral_BSA_m2_m3soil"].sum()


fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# left: total BSA vs grain diameter, one line per roughness rule
diam_sweep = np.logspace(0.5, 3, 40)  # ~3 to 1000 um
for rule in rules:
    vals = [total_bsa(d, 6.0, rule) for d in diam_sweep]
    axes[0].plot(diam_sweep, vals, label=rule, color=rule_colors[rule], linewidth=2)
# axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("grain diameter [µm]")
axes[0].set_ylabel("total BSA [m² / m³ soil]")
axes[0].set_title("BSA vs. grain size, by roughness rule\n(6 t/ha, 0.2 m till depth)")
axes[0].legend(frameon=False, fontsize=9)

# right: total BSA vs application rate (linear, at fixed grain size + roughness rule)
apprate_sweep = np.linspace(0.5, 60, 40)
vals = [total_bsa(200.0, a, "constant", const_rf=10) for a in apprate_sweep]
axes[1].plot(apprate_sweep, vals, color=CAT4[0], linewidth=2)
axes[1].set_xlabel("application rate [t/ha]")
axes[1].set_ylabel("total BSA [m² / m³ soil]")
axes[1].set_title("BSA vs. application rate\n(200 µm, const RF, 0.2 m till depth)")

fig.tight_layout()
plt.show()

## 4. Depth distribution

So far we've assumed that mineral conditions in the feedstock mix depth are uniform, so within the grid cells that contain feedstock there is no need to define a depth dependence. 

However, some numerical solvers might struggle with an abrupt drop in feedstock below the mixing depth, and alternative mixing profiles might offer more realistic process-representation in some systems. We address this by building a relative depth profile — a dimensionless curve whose area sums to the uniform case (in its volume fraction and BSA). This conserves volume and mass, effectively redistributing rock in the soil profile.

For now, the only shape beyond the uniform case is sigmoidal. It reads in a single paramter value that sets the steepness of the curve.

In [ ]:
def depth_profile_shape(distribution, tilldepth_m, n_cells, cell_thickness_m, sigmoidal_steepness=0.4,
                        beta_alpha=1.1, beta_beta=1.1, beta_depth_factor=1.0):
    """Relative depth profile (area-matched to the uniform case) as a DataFrame
    with columns depth_dx (1-indexed cell number) and relative_value.
    beta_alpha, beta_beta : float
        Shape parameters for the beta distribution.
    beta_depth_factor : float
        Maximum depth of the beta distribution relative to tilldepth_m.
        For example, 1.5 means the profile tapers to zero at
        1.5 * tilldepth_m.
    """
    n_till_cells = round(tilldepth_m / cell_thickness_m)
    depth_dx = np.arange(n_cells) + 1
    # Cell-center depths
    depth_m = (depth_dx - 1) * cell_thickness_m

    def uniform():
        return np.concatenate([np.ones(n_till_cells), np.zeros(n_cells - n_till_cells)])

    target_sum = uniform().sum()
    def scale_to_uniform(raw):
        raw_sum = raw.sum()
        scaled = raw * target_sum / raw_sum
        return np.where(scaled < 1e-6, 0, scaled)

    def sigmoidal():
        raw = -1 / (1 + np.exp(sigmoidal_steepness * (depth_dx - n_till_cells)))
        return scale_to_uniform(raw)

    def beta():
        max_depth_m = beta_depth_factor * tilldepth_m
        x = depth_m / max_depth_m

        raw = np.zeros(n_cells)
        mask = (x > 0) & (x < 1)

        raw[mask] = x[mask] ** (beta_alpha - 1) * (1 - x[mask]) ** (beta_beta - 1)

        return scale_to_uniform(raw)

    if distribution == "uniform":
        relative_value = uniform()
    elif distribution == "sigmoidal":
        relative_value = sigmoidal()
    elif distribution == "beta":
        relative_value = beta()
    else:
        raise ValueError(f"unknown distribution: {distribution!r}")

    return pd.DataFrame({"depth_dx": depth_dx, "relative_value": relative_value})


def apply_depth_profile(mineral_props_df, profile_df):
    """Broadcast each mineral's well-mixed vol_frac_in_soil / mass / BSA across depth
    using a relative depth profile. mineral_SSA_m2_kg is an intrinsic per-mass
    property, so it's carried through unscaled (same value at every depth).
    Returns one long-form DataFrame (MINERAL x depth_dx).
    """
    depth_list = []
    for _, row in mineral_props_df.iterrows():
        dfmin = profile_df.copy()
        dfmin["MINERAL"] = row["MINERAL"]
        dfmin["vol_frac_in_soil"] = row["vol_frac_in_soil"] * dfmin["relative_value"]
        dfmin["mineral_mass_kg_m3soil"] = row["mineral_mass_kg_m3soil"] * dfmin["relative_value"]
        dfmin["mineral_BSA_m2_m3soil"] = row["mineral_BSA_m2_m3soil"] * dfmin["relative_value"]
        dfmin["mineral_SSA_m2_kg"] = row["mineral_SSA_m2_kg"]
        depth_list.append(dfmin)
    return pd.concat(depth_list, ignore_index=True)


# depth grid for the worked example: 1 cm cells, out to 0.5 m (2.5x the 0.2 m till depth)
CELL_THICKNESS_M = 0.01
N_CELLS = 60

profile_uniform = depth_profile_shape("uniform", tilldepth_m=0.3, n_cells=N_CELLS,
                                      cell_thickness_m=CELL_THICKNESS_M)
profile_sigmoidal = depth_profile_shape("sigmoidal", tilldepth_m=0.3, n_cells=N_CELLS,
                                        cell_thickness_m=CELL_THICKNESS_M, sigmoidal_steepness=0.4)
profile_beta = depth_profile_shape("beta", tilldepth_m=0.3, n_cells=N_CELLS,
                                   cell_thickness_m=CELL_THICKNESS_M,
                                   beta_alpha=1.1, beta_beta=1.1, beta_depth_factor=1.0)
profile_beta.head()

In [ ]:
# compare the uniform shape against sigmoidal at a few steepness values
steepness_values = [0.1, 0.2, 0.5, 1.0]
steepness_colors = dict(zip(steepness_values, CAT4))

fig, ax = plt.subplots(figsize=(6.5, 5))
depth_m = profile_uniform["depth_dx"] * CELL_THICKNESS_M
ax.plot(profile_uniform["relative_value"], depth_m, color="#898781", linewidth=2, linestyle="--", label="uniform")
for k in steepness_values:
    prof = depth_profile_shape("sigmoidal", tilldepth_m=0.3, n_cells=N_CELLS, cell_thickness_m=CELL_THICKNESS_M, sigmoidal_steepness=k)
    ax.plot(prof["relative_value"], depth_m, color=steepness_colors[k], linewidth=2, label=f"sigmoidal, steepness={k}")

ax.axhline(0.3, color="#c3c2b7", linewidth=1, linestyle=":")
ax.text(ax.get_xlim()[1], 0.3, "  till depth", va="center", fontsize=9, color="#52514e")
ax.invert_yaxis()
ax.set_xlabel("relative value [-]")
ax.set_ylabel("depth [m]")
ax.set_title("Relative depth-distribution shapes\n(area-matched to the uniform profile)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# compare the uniform shape against beta
beta_values = [1.0, 1.1, 1.2, 1.8]
beta_colors = dict(zip(beta_values, CAT4))

fig, ax = plt.subplots(figsize=(6.5, 5))
depth_m = profile_uniform["depth_dx"] * CELL_THICKNESS_M
ax.plot(profile_uniform["relative_value"], depth_m, color="#898781", linewidth=2, linestyle="--", label="uniform")
for beta in beta_values:
    prof = depth_profile_shape("beta", tilldepth_m=0.3, n_cells=N_CELLS, cell_thickness_m=CELL_THICKNESS_M,
                               beta_alpha=beta, beta_beta=beta, beta_depth_factor=1.0)
    ax.plot(prof["relative_value"], depth_m, color=beta_colors[beta], linewidth=2,
            label=r"$\beta$="+f'{beta}'+r', $\alpha$='+f'{beta}')

ax.axhline(0.3, color="#c3c2b7", linewidth=1, linestyle=":")
ax.text(ax.get_xlim()[1], 0.3, "  till depth", va="center", fontsize=9, color="#52514e")
ax.invert_yaxis()
ax.set_xlabel("relative value [-]")
ax.set_ylabel("depth [m]")
ax.set_title("Relative depth-distribution shapes\n(area-matched to the uniform profile)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
depth_uniform = apply_depth_profile(mineral_props, profile_uniform)
depth_sigmoidal = apply_depth_profile(mineral_props, profile_sigmoidal)
profile_beta = depth_profile_shape("beta", tilldepth_m=0.3, n_cells=N_CELLS,
                                   cell_thickness_m=CELL_THICKNESS_M,
                                   beta_alpha=1.1, beta_beta=1.1, beta_depth_factor=1.0)
depth_beta = apply_depth_profile(mineral_props, profile_beta)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

# left: total BSA vs depth, uniform vs sigmoidal vs beta
for name, dfd, color in [("uniform", depth_uniform, CAT4[0]),
                         ("sigmoidal (steepness=0.4)", depth_sigmoidal, CAT4[1]),
                         ("beta (beta=1.5)", depth_beta, CAT4[2])]:
    tot = dfd.groupby("depth_dx")["mineral_BSA_m2_m3soil"].sum()
    axes[0].plot(tot.values, tot.index * CELL_THICKNESS_M, color=color, linewidth=2, label=name)
axes[0].set_xlabel("total BSA [m² / m³ soil]")
axes[0].set_ylabel("depth [m]")
axes[0].set_title("Total mineral BSA vs. depth")
axes[0].legend(frameon=False)

# right: per-mineral BSA depth profile, sigmoidal case (stacked, depth on the shared inverted y-axis)
pivot = depth_beta.pivot(index="depth_dx", columns="MINERAL", values="mineral_BSA_m2_m3soil")
pivot = pivot[order.values]  # keep basalt's wt%-descending mineral order from section 3
depths = pivot.index * CELL_THICKNESS_M
stack_colors = CAT6[: pivot.shape[1]]
cum_left = np.zeros(len(pivot))
for mineral, color in zip(pivot.columns, stack_colors):
    cum_right = cum_left + pivot[mineral].values
    axes[1].fill_betweenx(depths, cum_left, cum_right, color=color, label=mineral)
    cum_left = cum_right
axes[1].set_xlabel("mineral BSA [m² / m³ soil]")
axes[1].set_title("Per-mineral BSA vs. depth (beta)")
axes[1].legend(frameon=False, fontsize=8, loc="lower right")

axes[0].invert_yaxis()  # shared y-axis: one invert call flips both panels
fig.tight_layout()
plt.show()

## 5. Bringing it together 

We now combine the components into a single function, `initialize_feedstock_conditions`, that nests the feedstock mineral properties and depth profile calculations.

In [ ]:
def initialize_feedstock_conditions(
    composition, mineral_properties, dens_feedstock_g_cm3, appRate_ton_ha,
    tilldepth_m, diam_um, roughness_option="smooth", const_rf=None,
    n_cells=50, cell_thickness_m=0.01,
    dust_distribution="uniform", dust_sigmoidal_steepness=0.4,
    beta_alpha=1.1, beta_beta=1.1, beta_depth_factor=1.0,
):
    """Compute depth-resolved initial feedstock conditions (volume fraction and
    bulk surface area per mineral, per depth cell) for an ERW amendment.

    Nests the section 1-4 steps:
      1. composition_to_df       -- {mineral: wt_frac} + molar properties -> DataFrame
      2. feedstock_mineral_properties -- per-mineral SSA / BSA / volume fraction,
                                          assuming instant uniform mixing to tilldepth_m
      3. depth_profile_shape     -- relative depth-distribution shape (uniform/sigmoidal)
      4. apply_depth_profile     -- broadcast (2) across depth using (3)

    n_cells / cell_thickness_m define the depth grid the profile is evaluated on
    (n_cells cells, each cell_thickness_m thick). This assumes a *regular* grid
    (every cell the same thickness) -- it does not currently support an
    irregular/variable-thickness grid (e.g. matching a CrunchFlow setup with
    finer cells near the surface). That's a reasonable extension if/when it's
    needed, but isn't built in here.

    const_rf: fixed roughness factor, required when roughness_option="constant"; passed to feedstock_mineral_properties

    Returns (depth_resolved_df, mineral_summary_df, diagnostics).
    """
    feedstock_df = composition_to_df(composition, mineral_properties)
    mineral_summary, diagnostics = feedstock_mineral_properties(
        feedstock_df, dens_feedstock_g_cm3, appRate_ton_ha, tilldepth_m, diam_um, roughness_option, const_rf,
    )
    profile = depth_profile_shape(
        dust_distribution, tilldepth_m, n_cells, cell_thickness_m, dust_sigmoidal_steepness,
        beta_alpha, beta_beta, beta_depth_factor
    )
    depth_resolved = apply_depth_profile(mineral_summary, profile)
    return depth_resolved, mineral_summary, diagnostics


# --- worked example: same basalt case as sections 1-4, in one call ---
depth_resolved, mineral_summary, diagnostics = initialize_feedstock_conditions(
    composition               =FEEDSTOCK_COMPOSITION,
    mineral_properties        =MINERAL_PROPERTIES,
    dens_feedstock_g_cm3      = FEEDSTOCK_DENSITY,
    appRate_ton_ha            = application_rate,
    tilldepth_m               = 0.3,
    diam_um                   = diameter,
    roughness_option          = "constant",
    const_rf                  = roughness,
    n_cells                   = 50,
    cell_thickness_m          = 0.01,
    dust_distribution         = "beta",
    dust_sigmoidal_steepness  = 0.4,
    beta_alpha                = 1.1,
    beta_beta                 = 1.1,
    beta_depth_factor         = 1.0,
)
print(diagnostics)
print(depth_resolved.head())

# Save to S3
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'
outfile = f'forsterite_{application_rate}tha_{diameter}um_{roughness}rf.csv'
depth_resolved.to_csv(f'{s3_input_path}/{outfile}', index=False)

In [ ]:
# Check that depth_resolved adds up to the correct application rate
# Calculate total mass per unit soil area by integrating mineral_mass_kg_m3soil over depth
total_mass_kg_m2 = (depth_resolved['mineral_mass_kg_m3soil'] * CELL_THICKNESS_M).sum()

# Convert kg/m2 to tonnes/hectare (1 ha = 10,000 m2, 1 tonne = 1000 kg)
calculated_application_rate = total_mass_kg_m2 * 10000 / 1000

print(f"Input application rate: {application_rate} t/ha")
print(f"Calculated application rate from depth_resolved: {calculated_application_rate:.2f} t/ha")
print(f"Difference: {abs(calculated_application_rate - application_rate):.4f} t/ha")


### Swapping to a single-mineral feedstock

Everything above handles any number of minerals, including one. Here's a real
single-mineral case — agricultural lime (aglime), pure calcite — applied at a
much higher rate typical of liming (35 t/ha) with finer, ground grains (150 µm):

In [ ]:
AGLIME_COMPOSITION = {"Calcite": 1.0}
AGLIME_PROPERTIES = {"Calcite": {"molVolume_cm3_mol": 36.934, "molMass_g_mol": 100.0872}}

aglime_depth, aglime_summary, aglime_diag = initialize_feedstock_conditions(
    composition=AGLIME_COMPOSITION,
    mineral_properties=AGLIME_PROPERTIES,
    dens_feedstock_g_cm3=100.0872 / 36.934,  # pure calcite: density follows directly from its own molar properties
    appRate_ton_ha=6.0,
    tilldepth_m=0.2,
    diam_um=200.0,
    roughness_option="constant",
    const_rf=10,
    dust_distribution="sigmoidal",
    dust_sigmoidal_steepness=0.4,
)
aglime_summary

In [ ]:
# final comparison: total reactive surface area vs. depth, basalt (6 minerals) vs. aglime (1 mineral)
fig, ax = plt.subplots(figsize=(6.5, 5))
for name, dfd, color in [
    ("basalt, 6 t/ha, 200 µm", depth_resolved, CAT4[0]),
    ("aglime (calcite), 6 t/ha, 200 µm", aglime_depth, CAT4[1]),
]:
    tot = dfd.groupby("depth_dx")["mineral_BSA_m2_m3soil"].sum()
    ax.plot(tot.values, tot.index * CELL_THICKNESS_M, color=color, linewidth=2, label=name)
ax.invert_yaxis()
ax.set_xlabel("total BSA [m² / m³ soil]")
ax.set_ylabel("depth [m]")
ax.set_title("initialize_feedstock_conditions: multi- vs. single-mineral feedstock\n(both sigmoidal, steepness=0.4)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()